In [1]:
import pandas as pd

In [ ]:
#read
df=pd.read_csv("insurance_claims.csv", parse_dates=["policy_bind_date", "incident_date"], index_col="policy_number")

#check duplicates
df.duplicated().sum() 
df['authorities_contacted']=df['authorities_contacted'].fillna("Police")

#ensure consistency on column names
rename ={'capital-gains':'capital_gains','capital-loss': 'capital_loss'}
df.rename(columns=rename, inplace=True)

#Rename Strings
df["collision_type"]=df["collision_type"].apply(lambda x: "No Collision" if str(x)=="?" else x)
df["police_report_available"]=df["police_report_available"].apply(lambda x: "Unknown" if str(x)=="?" else x)
df["property_damage"]=df["property_damage"].apply(lambda x: "Unknown" if str(x)=="?" else x)

# Drop

In [3]:
# drop column as 100% nan-values
# drop umbrella limits as 80% of values are zeros, std = 1.1 Mio
df.drop(['months_as_customer',  'umbrella_limit',  'total_claim_amount', '_c39'],axis=1,inplace= True)

In [4]:
df.isna().sum().sum()

np.int64(0)

In [5]:
# #Type df.drop stuff
# for col in df.columns:
#     print(f"df.drop('{col}',axis=1,inplace=True)")

In [6]:
df.drop('age',axis=1,inplace=True)
df.drop('policy_bind_date',axis=1,inplace=True)
df.drop('policy_state',axis=1,inplace=True)
df.drop('policy_csl',axis=1,inplace=True)
df.drop('policy_deductable',axis=1,inplace=True)
df.drop('policy_annual_premium',axis=1,inplace=True)
df.drop('insured_zip',axis=1,inplace=True)
df.drop('insured_sex',axis=1,inplace=True)
df.drop('insured_education_level',axis=1,inplace=True)
df.drop('insured_occupation',axis=1,inplace=True)
df.drop('insured_relationship',axis=1,inplace=True)
df.drop('capital_gains',axis=1,inplace=True)
df.drop('capital_loss',axis=1,inplace=True)
df.drop('incident_date',axis=1,inplace=True)
df.drop('incident_city',axis=1,inplace=True)
df.drop('incident_location',axis=1,inplace=True)
df.drop('incident_hour_of_the_day',axis=1,inplace=True)
df.drop('number_of_vehicles_involved',axis=1,inplace=True)
df.drop('property_damage',axis=1,inplace=True)
df.drop('bodily_injuries',axis=1,inplace=True)
df.drop('witnesses',axis=1,inplace=True)
df.drop('police_report_available',axis=1,inplace=True)
# df.drop('injury_claim',axis=1,inplace=True)
# df.drop('property_claim',axis=1,inplace=True)
# df.drop('vehicle_claim',axis=1,inplace=True)
df.drop('auto_make',axis=1,inplace=True)
df.drop('auto_model',axis=1,inplace=True)
df.drop('auto_year',axis=1,inplace=True)


In [7]:
df.columns

Index(['insured_hobbies', 'incident_type', 'collision_type',
       'incident_severity', 'authorities_contacted', 'incident_state',
       'injury_claim', 'property_claim', 'vehicle_claim', 'fraud_reported'],
      dtype='object')

In [8]:
from sklearn.preprocessing import LabelEncoder
LabelEnc = LabelEncoder()
df['fraud_reported'] = LabelEnc.fit_transform(df['fraud_reported'])

# Train Test Split

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('fraud_reported', axis=1), df['fraud_reported'], test_size=0.2, random_state=42)

# Encoding


In [10]:
from sklearn.preprocessing import OneHotEncoder

enc=OneHotEncoder(handle_unknown='ignore')
X_train_enc=enc.fit_transform(X_train[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
#dataframe
X_train_enc=pd.DataFrame(X_train_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))

#reset index
X_train_enc.reset_index(drop=True, inplace=True)
X_train.reset_index(drop=True, inplace=True)

#concatenate
X_train=pd.concat([X_train, X_train_enc], axis=1)

#drop columns
X_train.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)

# same for test data
X_test_enc=enc.transform(X_test[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
X_test_enc=pd.DataFrame(X_test_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))
X_test_enc.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
X_test=pd.concat([X_test, X_test_enc], axis=1)
X_test.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)


In [11]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

minmax_scaler = MinMaxScaler()
rbst_scaler = RobustScaler()

minmax_col = [ "injury_claim",'vehicle_claim']
rbst_col = ['property_claim']


for column in minmax_col:
    X_train[column] = minmax_scaler.fit_transform(X_train[[column]])
    X_test[column] = minmax_scaler.fit_transform(X_test[[column]])
    
for column in rbst_col:
    X_train[column] = rbst_scaler.fit_transform(X_train[[column]])
    X_test[column] = rbst_scaler.transform(X_test[[column]])
    

In [12]:
#df.to_csv('insurance_claims_clean.csv')
X_train.to_csv("X_train_1.csv")
X_test.to_csv("X_test_1.csv")